# Netlify Deployment Fix: Optimize Serverless Bundle Size

## Problem
- Next.js serverless handler exceeds Netlify's **250 MB limit**
- All API routes bundled together with `@aws-sdk/*` and `@prisma/client`
- 6 failed deployments so far

## Solution
Migrate heavy API routes to dedicated Netlify Functions

## Step 1: Diagnose Bundle Size

Run locally to check what's consuming space:

```bash
npm run build
du -sh .netlify/functions/__netlify-server-handler
du -h .netlify/functions/__netlify-server-handler/node_modules | sort -h | tail -20
```

This will show which packages are largest (expect @prisma and @aws-sdk to be at top)

## Step 2: API Routes to Migrate

### AWS S3 Routes (Remove after migration)
- `src/app/api/galleries/[accessCode]/download/[photoId]` → Gallery downloads
- `src/app/api/debug-s3` → S3 debug endpoint
- `src/app/api/favicon/upload` → Favicon uploads

### Prisma Routes (Keep lightweight or move to Netlify Functions)
- `src/app/api/bookings` → Uses Prisma
- `src/app/api/blog/[id]` → Uses Prisma
- `src/app/api/settings/*` → Uses Prisma
- `src/app/api/menu` → Uses Prisma

### Strategy
1. **Keep lightweight**: Settings, menu - minimal Prisma queries
2. **Move to Netlify Functions**: Heavy S3 operations, complex queries

## Step 3: Alternative - Use Prisma Data Proxy (Recommended)

**This is the EASIEST fix if you have Prisma Accelerate subscription:**

1. Go to https://www.prisma.io/data-platform/accelerate
2. Create a Data Proxy connection string
3. Update `.env.production`:
   ```
   DATABASE_URL="prisma://accelerate.prisma-data.net/?api_key=YOUR_KEY"
   ```
4. This removes the need to bundle Prisma engine binary entirely
5. Rebuild and deploy - bundle size drops dramatically

**Prisma Team recommendation**: Data Proxy is the standard way to deploy to serverless.

## Step 4: Next.js Config Optimization

Update `next.config.mjs` to ensure proper bundling:

```javascript
const nextConfig = {
  output: 'standalone',
  images: { unoptimized: true },
  webpack: (config, { isServer }) => {
    if (!isServer) {
      // Browser builds don't need Prisma
      config.externals = [
        ...(config.externals || []),
        '@prisma/client',
      ];
    }
    return config;
  },
};
```

## Recommended Action Plan

### Option A: Quick Fix (Use Prisma Data Proxy) ⭐ RECOMMENDED
1. Setup Prisma Accelerate (free tier available)
2. Update DATABASE_URL in `.env.production`
3. Rebuild and deploy
4. **Expected result**: Bundle drops from 250MB+ to <100MB

### Option B: Manual Migration (Longer but more control)
1. Move S3 operations to `netlify/functions/`
2. Keep Prisma routes light (read-only)
3. Update frontend to call new endpoints
4. Delete old API routes
5. Rebuild and deploy

### Option C: Hybrid Approach
- Use Prisma Data Proxy for database ops
- Keep most API routes in Next.js (lightweight)
- Only move heavy S3 operations if still over limit